# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata

print(f"{metadata_obj.name}: {metadata_obj.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Identify the record sets defined in the Croissant schema and list their `@id`s.

In [ ]:
# List record sets and their fields using their @id references.
record_sets = []

# If dataset.metadata.recordSet is empty, try to enumerate distributions or sources.
if hasattr(metadata_obj, 'recordSet') and metadata_obj.recordSet:
    for rs in metadata_obj.recordSet:
        if hasattr(rs, '@id'):
            record_sets.append(rs['@id'])
        elif hasattr(rs, 'id_'):
            record_sets.append(rs.id_)
        else:
            print('Could not resolve @id for record set.')
    print('Record Sets:')
    print(record_sets)
else:
    # For manual exploration, infer recordSet @ids if possible
    print('No record sets found in metadata. Attempting to enumerate data sources/distributions...')
    dist_ids = []
    if hasattr(metadata_obj, 'distribution'):
        for dist in metadata_obj.distribution:
            if hasattr(dist, '@id'):
                dist_ids.append(dist['@id'])
            elif hasattr(dist, 'id_'):
                dist_ids.append(dist.id_)
        print('Distributions (@id):')
        print(dist_ids)
    else:
        print('No distributions found.')
    # For the purposes of the notebook, manually define record set @ids (example)
    record_sets = [
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2-table-1',
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2-table-2',
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2-table-3'
    ]
    print('Manually added record set @ids:')
    print(record_sets)
# List sample records from each record set
for rs_id in record_sets:
    try:
        print(f'--- Sample records from record set @id: {rs_id} ---')
        for x in dataset.records(record_set=rs_id):
            print(x)
            break  # Show only the first sample
    except Exception as e:
        print(f'Could not load record set {rs_id}: {e}')

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
Use the record set and field `@id`s identified above.

In [ ]:
# Extract data from each record set (@id)
dataframes = {}
for record_set in record_sets:
    try:
        records = list(dataset.records(record_set=record_set))
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f'Columns in DataFrame for record set {record_set}:')
        print(df.columns.tolist())
        print(df.head())
    except Exception as e:
        print(f'Could not load records for {record_set}: {e}')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
This section includes operations such as removing outliers, transforming numeric distributions, or grouping by key attributes.

In [ ]:
# Choose a record set and numeric field for EDA
# Example: Table 1 - Age at Diagnosis
table1_id = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2-table-1'
df1 = dataframes.get(table1_id, pd.DataFrame())

# List columns for reference
print('Table 1 columns:', df1.columns.tolist())

# Assume the age column is present and reference its @id
age_field_id = '@id:age_at_diagnosis'

# If actual column name differs, use df1.columns to check actual name
if 'age_at_diagnosis' in df1.columns:
    age_col = 'age_at_diagnosis'
else:
    # Fallback: use first numeric column
    numeric_cols = df1.select_dtypes(include=['int', 'float']).columns.tolist()
    age_col = numeric_cols[0] if numeric_cols else df1.columns[0]

# Filter records with age > threshold
threshold = 10
filtered_df = df1[df1[age_col] > threshold]
print(f"Filtered records with age_at_diagnosis > {threshold}:")
print(filtered_df.head())

# Normalize age
filtered_df[f"{age_col}_normalized"] = (filtered_df[age_col] - filtered_df[age_col].mean()) / filtered_df[age_col].std()
print(f"Normalized age_at_diagnosis for filtered records:")
print(filtered_df[[age_col, f"{age_col}_normalized"]].head())

# Group by categorical field (e.g., presence of infertility, referenced by @id)
infertility_field_id = '@id:infertility'
group_field = 'infertility' if 'infertility' in df1.columns else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[age_col].mean().reset_index()
    print(f"Grouped data by {group_field} (mean age_at_diagnosis):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Plot histogram of age at diagnosis
if not df1.empty and age_col in df1.columns:
    plt.figure(figsize=(6,4))
    df1[age_col].hist(bins=5)
    plt.title('Age at Diagnosis Distribution')
    plt.xlabel('Age')
    plt.ylabel('Frequency')
    plt.show()

# Plot infertility status vs age
if group_field:
    plt.figure(figsize=(6,4))
    grouped_df.plot.bar(x=group_field, y=age_col, legend=False, ax=plt.gca())
    plt.title('Mean Age at Diagnosis by Infertility Status')
    plt.xlabel('Infertility')
    plt.ylabel('Mean Age')
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated the use of `mlcroissant` to load and analyze the FAIR^2 dataset containing clinical, hormonal, imaging, and genetic data for three female patients. Key steps included parsing metadata, extracting record sets using their `@id`s, performing basic EDA, and visualizing numeric and categorical features. Due to the small sample size, findings serve illustrative purposes and highlight the richness and heterogeneity in phenotype-genotype relationships. For future work, integrating larger cohorts and additional features may further enhance insights.